# Developing and Comparing Sequential and Distributed Algorithms with maia

## Introduction

Using **Maia**, a Python/C++ library for working with CGNS meshes in parallel with MPI. The idea of this activity is to write an algorithm in a "distributed way", ie. operating on a distributed tree.

The objective is to compare three methods for calculating the geometric centers of mesh cells from a CGNS tree:
1. A sequential algorithm,
2. A parallel algorithm on a partitioned tree,
3. A fully distributed algorithm.

You'll discover how to: 
- Load CGNS trees in distributed, sequential, partitioned version and compare the execution times,
- Apply field calculations across unstructured meshes,
- Utilize Maia's exchange and indexing tools to perform tasks in parallel,
- Examine the trade-offs between each method's design and performance.

These exercises, which are particularly appropriate for individuals wishing to expand or enhance parallel mesh processing pipelines, demonstrates a common development task when developing new functionalities in Maia.

<script>
window.addEventListener('load', function() {
  var codeCells = document.querySelectorAll('.code_cell .input');
  codeCells.forEach(cell => {
    cell.style.display = "none";
  });
});
</script>


In [1]:
import time
import numpy as np
from mpi4py import MPI
comm = MPI.COMM_WORLD

In [2]:
import maia
import maia.pytree as PT

In [3]:
FILENAME = '/home/jovyan/Public/maia_training/MESHES/tetra10.hdf'

In [4]:
def _generate_case():
    from pathlib import Path
    if not Path(FILENAME).exists():
        tree = maia.factory.generate_dist_block(11, 'TETRA_4', comm)
        maia.io.dist_tree_to_file(tree, FILENAME, comm)

In [5]:
class DIndexer:
    def __init__(self, distri, indices, comm):
        from maia.transfer import protocols as EP
        self.btp = EP.GlobalIndexer(distri, indices-1, comm)

    def take(self, data_in):
        return self.btp.Take(data_in)

In [6]:
def compute_cc_seq(tree):
    zone = PT.get_node_from_label(tree, 'Zone_t')

    cx, cy, cz = PT.Zone.coordinates(zone)
    connec = PT.get_node_from_name(zone, 'ElementConnectivity')[1]

    n_elem = connec.size // 4
    connec_idx = 4*np.arange(n_elem+1)

    mean_x = np.add.reduceat(np.take(cx, connec-1), connec_idx[:-1]) / 4
    mean_y = np.add.reduceat(np.take(cy, connec-1), connec_idx[:-1]) / 4
    mean_z = np.add.reduceat(np.take(cz, connec-1), connec_idx[:-1]) / 4

    PT.new_FlowSolution('Centers',
                        loc='CellCenter',
                        fields={'CCX' : mean_x, 'CCY' : mean_y, 'CCZ' : mean_z},
                        parent=zone)

In [7]:
def compute_cc_dist(dist_tree, comm):
    zone = PT.get_node_from_label(dist_tree, 'Zone_t')

    cx, cy, cz = PT.Zone.coordinates(zone)
    connec = PT.get_node_from_name(zone, 'ElementConnectivity')[1]

    vtx_distri = PT.maia.get_Distribution(zone, 'Vertex')[1]
    indexer = DIndexer(vtx_distri, connec, comm)
    

    dn_elem = connec.size // 4
    connec_idx = 4*np.arange(dn_elem+1)

    mean_x = np.add.reduceat(indexer.take(cx), connec_idx[:-1]) / 4
    mean_y = np.add.reduceat(indexer.take(cy), connec_idx[:-1]) / 4
    mean_z = np.add.reduceat(indexer.take(cz), connec_idx[:-1]) / 4
    
    PT.new_FlowSolution('Centers',
                        loc='CellCenter',
                        fields={'CCX' : mean_x, 'CCY' : mean_y, 'CCZ' : mean_z},
                        parent=zone)


In [8]:
#_generate_case()

In [9]:
# Sequential
#if comm.rank == 0:
    #tree = maia.io.read_tree(FILENAME)
    #compute_cc_seq(tree)
    #maia.io.write_tree(tree, 'sol.hdf')
    

In [10]:
#tree = maia.io.file_to_dist_tree(FILENAME, comm)
#ptree = maia.factory.partition_dist_tree(tree, comm)
#compute_cc_seq(ptree)
#maia.transfer.part_tree_to_dist_tree_all(tree, ptree, comm)
#maia.io.dist_tree_to_file(tree, 'sol.hdf', comm)


In [11]:
#tree = maia.io.file_to_dist_tree(FILENAME, comm)
#compute_cc_dist(tree, comm)
#maia.io.dist_tree_to_file(tree, 'sol.hdf', comm)


In [12]:
def inline_paragraph(lines):
    children = []
    for line in lines:
        children.extend([line, v.Html(tag='br')])
    return v.Html(tag='p', children=children)


In [13]:
import ipyvuetify as v
from IPython.display import display
import numpy as np  # utile si code exécuté

# Fonction de normalisation
def normaliser(code):
    return "\n".join([
        line.strip()
        for line in code.strip().splitlines()
        if line.strip()
    ])

##################### STEP 1 ###############################
reference_code1 = """
import time
import numpy as np
from mpi4py import MPI
comm = MPI.COMM_WORLD
import maia
import maia.pytree as PT
"""

code_area1 = v.Textarea(
    label="Write your code here ...",
    outlined=True,
    rows=8,
    solo=True,
    v_model=""
)

bouton1 = v.Btn(children=["✅ Run"], color="success", dark=True)

res1 = v.Alert(
    type="info",
    children=["Result here"],
    outlined=True
)

def on_click1(widget, event, data):
    user_input = code_area1.v_model.strip()
    if not user_input:
        res1.children = ["⚠️ You didn't write anything"]
        res1.type = "warning"
    elif normaliser(user_input) == normaliser(reference_code1):
        res1.children = ["✅ Correct answer 🎉 !"]
        res1.type = "success"
    else:
        res1.children = ["Wrong answer 😞"]
        res1.type = "error"

bouton1.on_event("click", on_click1)

ui1 = v.Container(children=[
    v.Html(tag='h3', children=["Step 1 -- Import modules"]),
    inline_paragraph([
        "Maia operates in parallel! The so-called COMM_WORLD communicator must be imported from mpi4py first because practically all functions require an MPI communicator.",
        "We need to import the COMM_WORLD communicator from mpi4py, as well as numpy and time.",
        "Open the documentation that will be useful for this TP first: /Maia/1.3/index.html or https://onera.github.io.",
        "Take a brief look at the structure of the various modules (User Manual) and the definition of the parallel CGNS tree (Introduction > Maia CGNS Tree).",
        "Then, import `maia` and the module `pytree` from maia."
    ]),
    code_area1,
    bouton1,
    res1
])

display(ui1)

##################### STEP 2 ###############################
reference_code2 = """
def compute_cc_seq(tree):
    zone = PT.get_node_from_label(tree, 'Zone_t')

    cx, cy, cz = PT.Zone.coordinates(zone)
    connec = PT.get_node_from_name(zone, 'ElementConnectivity')[1]

    n_elem = connec.size // 4
    connec_idx = 4*np.arange(n_elem+1)

    mean_x = np.add.reduceat(np.take(cx, connec-1), connec_idx[:-1]) / 4
    mean_y = np.add.reduceat(np.take(cy, connec-1), connec_idx[:-1]) / 4
    mean_z = np.add.reduceat(np.take(cz, connec-1), connec_idx[:-1]) / 4

    PT.new_FlowSolution('Centers',
                        loc='CellCenter',
                        fields={'CCX' : mean_x, 'CCY' : mean_y, 'CCZ' : mean_z},
                        parent=zone)
"""

code_area2 = v.Textarea(
    label="Write your code here ...",
    outlined=True,
    rows=12,
    solo=True,
    v_model=""
)

bouton2 = v.Btn(children=["✅ Run"], color="success", dark=True)

res2 = v.Alert(
    type="info",
    children=["Result here"],
    outlined=True
)

def on_click2(widget, event, data):
    user_input = code_area2.v_model.strip()
    if not user_input:
        res2.children = ["⚠️ You didn't write anything"]
        res2.type = "warning"
    elif normaliser(user_input) == normaliser(reference_code2):
        res2.children = ["✅ Correct answer 🎉 !"]
        res2.type = "success"
    else:
        res2.children = ["Wrong answer 😞"]
        res2.type = "error"

bouton2.on_event("click", on_click2)
ui2 = v.Container(children=[
    v.Html(tag='h3', children=[" Step 2 -- Compute cell centers in a CGNS zone"]),
    inline_paragraph(["Create a function called compute_cc_seq(tree) that determines each cell's geometric center, or centroid, within a CGNS zone.",
                              "Within the function:",
                              "Use PT.get_node_from_label(tree, 'Zone_t') to retrieve the zone node from the tree.",
                              "Use PT.Zone.coordinates(zone) to extract the node coordinates cx, cy, and cz.",
                              "Use PT.get_node_from_name(zone, 'ElementConnectivity')[1] to obtain the cell connectivity array.",
                              "Assume that every cell is a quadruple consisting of four nodes.",
                              "Use connec_idx = 4 * np.arange(n_elem+1) to calculate the number of elements and construct the connectivity index array.",
                              "To determine the average coordinates (mean_x, mean_y, and mean_z) for every cell, use np.add.reduceat and np.take.",
                              "At the CellCenter location, create a new FlowSolution_t node called 'Centers'.",
                              "Enter the calculated centroids in the 'CCX,' 'CCY,' and 'CCZ' fields.",
                              "Connect this node to the zone."]),
    code_area2,
    bouton2,
    res2
])

display(ui2)

##################### STEP 3 ###############################
reference_code3 = """
def compute_cc_dist(dist_tree, comm):
    zone = PT.get_node_from_label(dist_tree, 'Zone_t')
    cx, cy, cz = PT.Zone.coordinates(zone)
    connec = PT.get_node_from_name(zone, 'ElementConnectivity')[1]
    vtx_distri = PT.maia.get_Distribution(zone, 'Vertex')[1]
    indexer = DIndexer(vtx_distri, connec, comm)
    dn_elem = connec.size // 4
    connec_idx = 4*np.arange(dn_elem+1)
    mean_x = np.add.reduceat(indexer.take(cx), connec_idx[:-1]) / 4
    mean_y = np.add.reduceat(indexer.take(cy), connec_idx[:-1]) / 4
    mean_z = np.add.reduceat(indexer.take(cz), connec_idx[:-1]) / 4
    PT.new_FlowSolution('Centers',
                        loc='CellCenter',
                        fields={'CCX' : mean_x, 'CCY' : mean_y, 'CCZ' : mean_z},
                        parent=zone)
"""

code_area3 = v.Textarea(
    label="Write your code here ...",
    outlined=True,
    rows=12,
    solo=True,
    v_model=""
)

bouton3 = v.Btn(children=["✅ Run"], color="success", dark=True)

res3 = v.Alert(
    type="info",
    children=["Result here"],
    outlined=True
)

def on_click3(widget, event, data):
    user_input = code_area3.v_model.strip()
    if not user_input:
        res3.children = ["⚠️ You didn't write anything"]
        res3.type = "warning"
    elif normaliser(user_input) == normaliser(reference_code3):
        res3.children = ["✅ Correct answer 🎉 !"]
        res3.type = "success"
    else:
        res3.children = ["Wrong answer 😞"]
        res3.type = "error"

bouton3.on_event("click", on_click3)

ui3 = v.Container(children=[
    v.Html(tag='h3', children=[" Step 3 -- Calculate cells' centers within a distributed CGNS tree"]),
    inline_paragraph(["To calculate cell centers (centroids) on a distributed CGNS tree, define the function compute_cc_dist(dist_tree, comm).",
                              "Within the function :",
                              "- Use PT.get_node_from_label(dist_tree, 'Zone_t') to retrieve the zone node from the distributed tree.",
                              "- Use PT.Zone.coordinates(zone) to extract the distributed node coordinates cx, cy, and cz.",
                              "- Use PT.get_node_from_name(zone, 'ElementConnectivity')[1] to obtain the element connectivity array.",
                              "- Use PT.maia.getDistribution(zone, 'Vertex')[1] to obtain the vertex distribution.",
                              "- Using the connectivity array, the MPI communicator comm, and the vertex distribution, create a DIndexer instance.",
                              "- Assume that every cell is a quad with four nodes.",
                              "- Construct the connectivity index array by calculating the number of elements: connec_idx = 4 * np.arange(dn_elem+1).",
                              "- To remap distributed coordinates to a local view, utilize the indexer.take() method.",
                              "- Use np.add.reduceat(...) / 4 to calculate the mean coordinates (mean_x, mean_y, and mean_z) for each cell.",
                              "- Make a new FlowSolution_t node called 'Centers' at the CellCenter location.",
                              "- Add the computed centroid coordinates as fields 'CCX', 'CCY', and 'CCZ'.",
                              "- Attach this solution node to the zone."]),
    code_area3,
    bouton3,
    res3
])

display(ui3)
####################### STEP 4.1 ################################


reference_code4_1 = """ _generate_case()"""

code_area4_1 = v.Textarea(
    label="Write your code here ...",
    outlined=True,
    rows=12,
    solo=True,
    v_model=""
)

bouton4_1 = v.Btn(children=["✅ Run"], color="success", dark=True)

res4_1 = v.Alert(
    type="info",
    children=["Result here"],
    outlined=True
)

def on_click4_1(widget, event, data):
    user_input = code_area4_1.v_model.strip()
    if not user_input:
        res4_1.children = ["⚠️ You didn't write anything"]
        res4._1type = "warning"
    elif normaliser(user_input) == normaliser(reference_code4_1):
        res4_1.children = ["✅ Correct answer 🎉 !"]
        res4_1.type = "success"
    else:
        res4_1.children = ["Wrong answer 😞"]
        res4_1.type = "error"

bouton4_1.on_event("click", on_click4_1)

ui4_1 = v.Container(children=[
    v.Html(tag='h3', children=[" Step 4.1 -- Call function _generate_case that saves a distributed tree into FILENAME"]),
    code_area4_1,
    bouton4_1,
    res4_1
])

display(ui4_1)


##################### STEP 4.2 ###############################
reference_code4_2 = """
if comm.rank == 0:
    tree = maia.io.read_tree('tetra10.hdf')
    compute_cc_seq(tree)
    maia.io.write_tree(tree, 'sol.hdf')
    PT.print_tree(tree)
"""

code_area4_2 = v.Textarea(
    label="Write your code here ...",
    outlined=True,
    rows=12,
    solo=True,
    v_model=""
)

bouton4_2 = v.Btn(children=["✅ Run"], color="success", dark=True)

res4_2 = v.Alert(
    type="info",
    children=["Result here"],
    outlined=True
)

def on_click4_2(widget, event, data):
    user_input = code_area4_2.v_model.strip()
    if not user_input:
        res4_2.children = ["⚠️ You didn't write anything"]
        res4_2.type = "warning"
    elif normaliser(user_input) == normaliser(reference_code4_2):
        res4_2.children = ["✅ Correct answer 🎉 !"]
        res4_2.type = "success"
    else:
        res4_2.children = ["Wrong answer 😞"]
        res4_2.type = "error"

bouton4_2.on_event("click", on_click4_2)

ui4_2 = v.Container(children=[
    v.Html(tag='h3', children=["Step 4.2 -- Use bigger meshes"]),
    inline_paragraph(["NB: You can try with bigger meshes. First, you need to generate it using:",
                              "dist_tree = maia.factory.generate_dist_block(101, 'TETRA_4', comm)",
                              "then save it using:",
                              "`maia.io.dist_tree_to_file(dist_tree, 'tetra100.hdf', comm)`"]),
    v.Html(tag='h3', children=["Step 4.2.1 -- Use the function compute_cc_seq"]),
    inline_paragraph(["Use the file `'tetra10.hdf'` to calculate cell centers with the function `compute_cc_seq`,",
                              "then save the resulting tree to a file named `'sol.hdf'`."]),
    code_area4_2,
    bouton4_2,
    res4_2
])


display(ui4_2)

##################### STEP 4.3 ###############################
reference_code4_3 = """
tree = maia.io.file_to_dist_tree('tetra10.hdf', comm)
ptree = maia.factory.partition_dist_tree(tree, comm)
compute_cc_seq(ptree)
maia.transfer.part_tree_to_dist_tree_all(tree, ptree, comm)
maia.io.dist_tree_to_file(tree, 'sol.hdf', comm)
PT.print_tree(tree)
"""

code_area4_3 = v.Textarea(
    label="Write your code here ...",
    outlined=True,
    rows=12,
    solo=True,
    v_model=""
)

bouton4_3 = v.Btn(children=["✅ Run"], color="success", dark=True)

res4_3 = v.Alert(
    type="info",
    children=["Result here"],
    outlined=True
)

def on_click4_3(widget, event, data):
    user_input = code_area4_3.v_model.strip()
    if not user_input:
        res4_3.children = ["⚠️ You didn't write anything"]
        res4_3.type = "warning"
    elif normaliser(user_input) == normaliser(reference_code4_3):
        res4_3.children = ["✅ Correct answer 🎉 !"]
        res4_3.type = "success"
    else:
        res4_3.children = ["Wrong answer 😞"]
        res4_3.type = "error"

bouton4_3.on_event("click", on_click4_3)

ui4_3 = v.Container(children=[
    v.Html(tag='h3', children=["Step 4.3 -- Use the function compute_cc_seq with a partitioned tree"]),
    v.Html(tag='p', children=["Now, we want to use the same file 'tetra10.hdf' with parallel partitioning."]),
    code_area4_3,
    bouton4_3,
    res4_3
])

display(ui4_3)

##################### STEP 4.4 ###############################
reference_code4_4 = """
tree = maia.io.file_to_dist_tree('tetra10.hdf', comm)
compute_cc_dist(tree, comm)
maia.io.dist_tree_to_file(tree, 'sol.hdf', comm)
PT.print_tree(tree)
"""

code_area4_4 = v.Textarea(
    label="Write your code here ...",
    outlined=True,
    rows=12,
    solo=True,
    v_model=""
)

bouton4_4 = v.Btn(children=["✅ Run"], color="success", dark=True)

res4_4 = v.Alert(
    type="info",
    children=["Result here"],
    outlined=True
)

def on_click4_4(widget, event, data):
    user_input = code_area4_4.v_model.strip()
    if not user_input:
        res4_4.children = ["⚠️ You didn't write anything"]
        res4_4.type = "warning"
    elif normaliser(user_input) == normaliser(reference_code4_4):
        res4_4.children = ["✅ Correct answer 🎉 !"]
        res4_4.type = "success"
    else:
        res4_4.children = ["Wrong answer 😞"]
        res4_4.type = "error"

bouton4_4.on_event("click", on_click4_4)

ui4_4 = v.Container(children=[
    v.Html(tag='h3', children=["Step 4.4 -- Use the function compute_cc_dist"]),
    inline_paragraph(["Now, we want to use the same file 'tetra10.hdf' with parallel distribution."]),
    code_area4_4,
    bouton4_4,
    res4_4
])

display(ui4_4)

Container(children=[Html(children=['Step 1 -- Import modules'], layout=None, tag='h3'), Html(children=['Maia o…

Container(children=[Html(children=[' Step 2 -- Compute cell centers in a CGNS zone'], layout=None, tag='h3'), …

Container(children=[Html(children=[" Step 3 -- Calculate cells' centers within a distributed CGNS tree"], layo…

Container(children=[Html(children=[' Step 4.1 -- Call function _generate_case that saves a distributed tree in…

Container(children=[Html(children=['Step 4.2 -- Use bigger meshes'], layout=None, tag='h3'), Html(children=['N…

Container(children=[Html(children=['Step 4.3 -- Use the function compute_cc_seq with a partitioned tree'], lay…

Container(children=[Html(children=['Step 4.4 -- Use the function compute_cc_dist'], layout=None, tag='h3'), Ht…